# 08 — Trade Candidate Mapping v0.1

Converts daily put/call signals into concrete naked SPX option trade candidates.

Rules finalized in this notebook:
- Calls and puts may use different tenors.
- Target expiry is `trade_date + selected_tenor`.
- Actual expiry is the cached listed expiry closest to the target expiry; ties choose the later expiry.
- Naked short put strike is ATM rounded down, then mapped to nearest listed put strike `<= SPX`.
- Naked short call strike is `+2SD` OTM using the VIX-style tenor vol and actual listed DTE, then mapped to nearest listed call strike `>= raw strike`.
- No long strikes.
- No risk sizing.
- No option pricing or P&L.


In [ ]:
# ============================================================
# 08_trade_candidate_mapping_v0_1
#
# Goal:
#   Convert daily put/call trade signals into naked option
#   trade candidates.
#
# Stage 1:
#   Load and validate signal files.
#
# Stage 2:
#   Build raw candidate rows with target expiry dates.
#
# No option pricing yet.
# No P&L yet.
# No risk sizing yet.
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RAW_DATA_DIR = DATA_DIR / "raw"
AUDIT_DIR = DATA_DIR / "audit"

DAILY_SIGNAL_SUMMARY_PARQUET = PROCESSED_DATA_DIR / "daily_trade_signal_summary_v0_1.parquet"
TRADE_SIGNAL_PANEL_PARQUET = PROCESSED_DATA_DIR / "trade_signal_panel_v0_1.parquet"

TRADE_CANDIDATE_RAW_CSV = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1_raw_targets.csv"
TRADE_CANDIDATE_RAW_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1_raw_targets.parquet"

print("Project root:", PROJECT_ROOT)
print("Daily signal summary exists:", DAILY_SIGNAL_SUMMARY_PARQUET.exists())
print("Trade signal panel exists:", TRADE_SIGNAL_PANEL_PARQUET.exists())
print("Raw data dir exists:", RAW_DATA_DIR.exists())

In [ ]:
# ============================================================
# Load daily signal summary from notebook 07
# ============================================================

daily_signal_df = pd.read_parquet(DAILY_SIGNAL_SUMMARY_PARQUET).copy()

required_daily_cols = [
    "trade_date",
    "spx_close",
    "spx_rsi_14",
    "signal_state",

    "selected_put_signal",
    "selected_put_tenor",
    "selected_put_tier",
    "selected_put_primary_vrp_signal",
    "selected_put_blended_z",
    "selected_put_3m_z",
    "selected_put_1y_z",
    "selected_put_implied_vol",
    "selected_put_trailing_rv",
    "selected_put_variance_ratio",
    "selected_put_vol_ratio",

    "selected_call_signal",
    "selected_call_tenor",
    "selected_call_primary_vrp_signal",
    "selected_call_blended_z",
    "selected_call_3m_z",
    "selected_call_1y_z",
    "selected_call_implied_vol",
    "selected_call_trailing_rv",
    "selected_call_variance_ratio",
    "selected_call_vol_ratio",

    "preferred_tenor",
]

missing_daily_cols = [c for c in required_daily_cols if c not in daily_signal_df.columns]

if missing_daily_cols:
    raise ValueError(f"Missing required daily columns: {missing_daily_cols}")

print("Daily signal rows:", len(daily_signal_df))
print("Date range:", daily_signal_df["trade_date"].min(), "to", daily_signal_df["trade_date"].max())

print("\nSignal state counts:")
display(daily_signal_df["signal_state"].value_counts().rename("count").to_frame())

print("\nSelected put signal dates:", daily_signal_df["selected_put_signal"].sum())
print("Selected call signal dates:", daily_signal_df["selected_call_signal"].sum())

display(daily_signal_df.tail(20))

In [ ]:
# ============================================================
# Date and strike helper functions
# ============================================================

def int_yyyymmdd_to_timestamp(date_int):
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d")


def timestamp_to_int_yyyymmdd(ts):
    return int(pd.Timestamp(ts).strftime("%Y%m%d"))


def add_calendar_days_to_trade_date(trade_date, tenor_days):
    """
    Target expiry date before mapping to listed option expiries.
    """
    if pd.isna(tenor_days):
        return np.nan

    trade_ts = int_yyyymmdd_to_timestamp(trade_date)
    target_expiry_ts = trade_ts + pd.Timedelta(days=int(tenor_days))

    return timestamp_to_int_yyyymmdd(target_expiry_ts)


def raw_atm_put_strike(spx_close):
    """
    Raw naked short put strike:
    ATM rounded down.
    Later we will map this to nearest listed put strike <= SPX.
    """
    if pd.isna(spx_close):
        return np.nan

    return np.floor(float(spx_close))


def raw_2sd_call_strike(spx_close, implied_vol, tenor_days):
    """
    Raw naked short call strike:
    SPX * (1 + 2 * implied_vol_decimal * sqrt(T))

    Uses selected tenor for now.
    After mapping to actual listed expiry, we may recalculate using actual DTE.
    """
    if pd.isna(spx_close) or pd.isna(implied_vol) or pd.isna(tenor_days):
        return np.nan

    implied_vol_decimal = float(implied_vol) / 100.0
    t = float(tenor_days) / 365.0

    return float(spx_close) * (1.0 + 2.0 * implied_vol_decimal * np.sqrt(t))


# Quick tests
test_trade_date = 20260625
test_spx = 7357.49
test_tenor = 30
test_iv = 15.0

print("Test target expiry:", add_calendar_days_to_trade_date(test_trade_date, test_tenor))
print("Test raw put strike:", raw_atm_put_strike(test_spx))
print("Test raw 2SD call strike:", raw_2sd_call_strike(test_spx, test_iv, test_tenor))

In [ ]:
# ============================================================
# Candidate mapping assumptions
# ============================================================

UNDERLYER = "SPX"

PUT_OPTION_STRUCTURE = "naked_short_put"
CALL_OPTION_STRUCTURE = "naked_short_call"

PUT_OPTION_RIGHT = "P"
CALL_OPTION_RIGHT = "C"

PUT_SHORT_STRIKE_RULE = "atm_rounded_down_nearest_listed_put_strike_lte_spx"
CALL_SHORT_STRIKE_RULE = "plus_2sd_otm_nearest_listed_call_strike_gte_raw"

EXPIRY_SELECTION_RULE = "closest_listed_expiry_to_target_later_on_tie"

DEFAULT_EXIT_RULE = "hold_to_expiry"

print("Underlyer:", UNDERLYER)
print("Put structure:", PUT_OPTION_STRUCTURE)
print("Call structure:", CALL_OPTION_STRUCTURE)
print("Expiry selection:", EXPIRY_SELECTION_RULE)
print("Exit rule:", DEFAULT_EXIT_RULE)

In [ ]:
# ============================================================
# Build raw trade candidate rows
# ============================================================

candidate_rows = []

for _, row in daily_signal_df.iterrows():
    trade_date = row["trade_date"]
    spx_close = row["spx_close"]

    # ----------------------------
    # Naked short put candidate
    # ----------------------------
    if bool(row["selected_put_signal"]):
        selected_tenor = row["selected_put_tenor"]
        target_expiry_date = add_calendar_days_to_trade_date(
            trade_date=trade_date,
            tenor_days=selected_tenor,
        )

        short_strike_raw = raw_atm_put_strike(spx_close)

        candidate_rows.append({
            "trade_date": trade_date,
            "underlyer": UNDERLYER,
            "trade_side": "put",
            "option_structure": PUT_OPTION_STRUCTURE,
            "option_right": PUT_OPTION_RIGHT,

            "selected_tenor": selected_tenor,
            "target_expiry_date": target_expiry_date,
            "expiry_selection_rule": EXPIRY_SELECTION_RULE,
            "exit_rule": DEFAULT_EXIT_RULE,

            "short_strike_rule": PUT_SHORT_STRIKE_RULE,
            "short_strike_raw": short_strike_raw,

            # These will be populated after we inspect/list actual chains.
            "selected_expiry_date": np.nan,
            "actual_dte": np.nan,
            "short_strike_selected": np.nan,

            "signal_state": row["signal_state"],
            "signal_tier": row["selected_put_tier"],

            "signal_primary_vrp": row["selected_put_primary_vrp_signal"],
            "signal_blended_z": row["selected_put_blended_z"],
            "signal_3m_z": row["selected_put_3m_z"],
            "signal_1y_z": row["selected_put_1y_z"],

            "signal_implied_vol": row["selected_put_implied_vol"],
            "signal_trailing_rv": row["selected_put_trailing_rv"],
            "signal_variance_ratio": row["selected_put_variance_ratio"],
            "signal_vol_ratio": row["selected_put_vol_ratio"],

            "spx_close": spx_close,
            "spx_rsi_14": row["spx_rsi_14"],
            "preferred_tenor": row["preferred_tenor"],
        })

    # ----------------------------
    # Naked short call candidate
    # ----------------------------
    if bool(row["selected_call_signal"]):
        selected_tenor = row["selected_call_tenor"]
        target_expiry_date = add_calendar_days_to_trade_date(
            trade_date=trade_date,
            tenor_days=selected_tenor,
        )

        short_strike_raw = raw_2sd_call_strike(
            spx_close=spx_close,
            implied_vol=row["selected_call_implied_vol"],
            tenor_days=selected_tenor,
        )

        candidate_rows.append({
            "trade_date": trade_date,
            "underlyer": UNDERLYER,
            "trade_side": "call",
            "option_structure": CALL_OPTION_STRUCTURE,
            "option_right": CALL_OPTION_RIGHT,

            "selected_tenor": selected_tenor,
            "target_expiry_date": target_expiry_date,
            "expiry_selection_rule": EXPIRY_SELECTION_RULE,
            "exit_rule": DEFAULT_EXIT_RULE,

            "short_strike_rule": CALL_SHORT_STRIKE_RULE,
            "short_strike_raw": short_strike_raw,

            # These will be populated after we inspect/list actual chains.
            "selected_expiry_date": np.nan,
            "actual_dte": np.nan,
            "short_strike_selected": np.nan,

            "signal_state": row["signal_state"],
            "signal_tier": "call_baseline",

            "signal_primary_vrp": row["selected_call_primary_vrp_signal"],
            "signal_blended_z": row["selected_call_blended_z"],
            "signal_3m_z": row["selected_call_3m_z"],
            "signal_1y_z": row["selected_call_1y_z"],

            "signal_implied_vol": row["selected_call_implied_vol"],
            "signal_trailing_rv": row["selected_call_trailing_rv"],
            "signal_variance_ratio": row["selected_call_variance_ratio"],
            "signal_vol_ratio": row["selected_call_vol_ratio"],

            "spx_close": spx_close,
            "spx_rsi_14": row["spx_rsi_14"],
            "preferred_tenor": row["preferred_tenor"],
        })

trade_candidate_raw_df = pd.DataFrame(candidate_rows)

print("Raw candidate rows:", len(trade_candidate_raw_df))
print("Unique trade dates:", trade_candidate_raw_df["trade_date"].nunique())

print("\nCandidates by side:")
display(trade_candidate_raw_df["trade_side"].value_counts().rename("count").to_frame())

print("\nCandidates by signal state:")
display(trade_candidate_raw_df["signal_state"].value_counts().rename("count").to_frame())

print("\nCandidates by side and signal state:")
display(
    trade_candidate_raw_df
    .groupby(["signal_state", "trade_side"])
    .size()
    .unstack(fill_value=0)
)

display(trade_candidate_raw_df.tail(40))

In [ ]:
# ============================================================
# Save raw target trade candidates
# ============================================================

trade_candidate_raw_df.to_csv(TRADE_CANDIDATE_RAW_CSV, index=False)
trade_candidate_raw_df.to_parquet(TRADE_CANDIDATE_RAW_PARQUET, index=False)

print("Saved raw candidate CSV:", TRADE_CANDIDATE_RAW_CSV)
print("Saved raw candidate parquet:", TRADE_CANDIDATE_RAW_PARQUET)

In [ ]:
# ============================================================
# Stage 3: Inspect local ThetaData chain cache
# ============================================================

RAW_CHAIN_CACHE_DIR = RAW_DATA_DIR / "thetadata_chains"

print("Raw chain cache directory:", RAW_CHAIN_CACHE_DIR)
print("Exists:", RAW_CHAIN_CACHE_DIR.exists())

if not RAW_CHAIN_CACHE_DIR.exists():
    raise FileNotFoundError(f"Raw chain cache directory not found: {RAW_CHAIN_CACHE_DIR}")

chain_files = sorted([
    p for p in RAW_CHAIN_CACHE_DIR.rglob("*")
    if p.is_file()
])

print("Number of files in raw chain cache:", len(chain_files))

if len(chain_files) == 0:
    raise FileNotFoundError(f"No files found in {RAW_CHAIN_CACHE_DIR}")

print("\nFirst 20 files:")
for p in chain_files[:20]:
    print(p.relative_to(RAW_CHAIN_CACHE_DIR))

print("\nLast 20 files:")
for p in chain_files[-20:]:
    print(p.relative_to(RAW_CHAIN_CACHE_DIR))

In [ ]:
# ============================================================
# Build cache file inventory
# ============================================================

file_inventory_rows = []

for p in chain_files:
    stat = p.stat()

    file_inventory_rows.append({
        "file_path": str(p),
        "relative_path": str(p.relative_to(RAW_CHAIN_CACHE_DIR)),
        "file_name": p.name,
        "suffix": p.suffix.lower(),
        "size_mb": stat.st_size / (1024 * 1024),
        "modified_time": pd.Timestamp(stat.st_mtime, unit="s"),
    })

cache_inventory_df = pd.DataFrame(file_inventory_rows)

print("Files:", len(cache_inventory_df))
print("\nSuffix counts:")
display(cache_inventory_df["suffix"].value_counts().rename("count").to_frame())

print("\nSize summary, MB:")
display(cache_inventory_df["size_mb"].describe())

display(cache_inventory_df.head(20))
display(cache_inventory_df.tail(20))

In [ ]:
# ============================================================
# Infer date tokens from file paths
# ============================================================

import re

def extract_yyyymmdd_tokens(text):
    tokens = re.findall(r"(20\d{6})", str(text))
    valid_tokens = []

    for token in tokens:
        try:
            pd.to_datetime(token, format="%Y%m%d")
            valid_tokens.append(int(token))
        except Exception:
            pass

    return valid_tokens


cache_inventory_df["yyyymmdd_tokens"] = cache_inventory_df["relative_path"].apply(
    extract_yyyymmdd_tokens
)

cache_inventory_df["num_date_tokens"] = cache_inventory_df["yyyymmdd_tokens"].apply(len)

display(
    cache_inventory_df[
        [
            "relative_path",
            "suffix",
            "size_mb",
            "num_date_tokens",
            "yyyymmdd_tokens",
        ]
    ].head(50)
)

print("Date-token count distribution:")
display(cache_inventory_df["num_date_tokens"].value_counts().sort_index().rename("count").to_frame())

# If there are at least two date tokens, often the first is trade date and second is expiry date.
cache_inventory_df["inferred_trade_date_from_path"] = cache_inventory_df["yyyymmdd_tokens"].apply(
    lambda x: x[0] if len(x) >= 1 else np.nan
)

cache_inventory_df["inferred_expiry_date_from_path"] = cache_inventory_df["yyyymmdd_tokens"].apply(
    lambda x: x[1] if len(x) >= 2 else np.nan
)

print("Inferred trade date range from path:")
print(
    cache_inventory_df["inferred_trade_date_from_path"].min(),
    "to",
    cache_inventory_df["inferred_trade_date_from_path"].max(),
)

print("Inferred expiry date range from path:")
print(
    cache_inventory_df["inferred_expiry_date_from_path"].min(),
    "to",
    cache_inventory_df["inferred_expiry_date_from_path"].max(),
)

In [ ]:
# ============================================================
# Read one sample cached chain file
# ============================================================

def convert_pickle_object_to_dataframe(obj):
    """
    Convert a pickle-loaded object into a DataFrame when possible.
    """
    if isinstance(obj, pd.DataFrame):
        return obj

    if isinstance(obj, dict):
        # If the dict contains one or more DataFrames, use the largest one.
        dataframe_items = {
            k: v for k, v in obj.items()
            if isinstance(v, pd.DataFrame)
        }

        if len(dataframe_items) > 0:
            largest_key = max(dataframe_items, key=lambda k: len(dataframe_items[k]))
            print("Pickle object is dict. Using DataFrame key:", largest_key)
            return dataframe_items[largest_key]

        # Otherwise try direct DataFrame conversion.
        try:
            return pd.DataFrame(obj)
        except Exception as e:
            raise ValueError(f"Pickle dict could not be converted to DataFrame: {e}")

    if isinstance(obj, list):
        try:
            return pd.DataFrame(obj)
        except Exception as e:
            raise ValueError(f"Pickle list could not be converted to DataFrame: {e}")

    raise ValueError(f"Unsupported pickle object type: {type(obj)}")


def read_cached_chain_file(file_path, nrows=None):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".parquet":
        df = pd.read_parquet(file_path)

    elif suffix == ".csv":
        df = pd.read_csv(file_path)

    elif suffix in [".txt", ".tsv"]:
        df = pd.read_csv(file_path)

    elif suffix == ".pkl":
        obj = pd.read_pickle(file_path)
        df = convert_pickle_object_to_dataframe(obj)

    else:
        raise ValueError(f"Unsupported cached chain file suffix: {suffix}")

    if nrows is not None:
        df = df.head(nrows)

    return df


# Prefer a non-empty file
sample_file_row = (
    cache_inventory_df[cache_inventory_df["size_mb"] > 0]
    .sort_values("size_mb", ascending=False)
    .head(1)
)

if sample_file_row.empty:
    raise ValueError("No non-empty cached chain files found.")

sample_file_path = sample_file_row.iloc[0]["file_path"]

print("Sample file:", sample_file_path)

sample_chain_df = read_cached_chain_file(sample_file_path)

print("Sample rows:", len(sample_chain_df))
print("Sample columns:")
for c in sample_chain_df.columns:
    print(" ", c)

print("\nDtypes:")
display(sample_chain_df.dtypes.rename("dtype").to_frame())

display(sample_chain_df.head(20))

In [ ]:
# ============================================================
# Inspect columns across multiple cached files
# ============================================================

sample_files_for_schema = (
    cache_inventory_df[cache_inventory_df["size_mb"] > 0]
    .sort_values("file_name")
    .head(20)
)

schema_rows = []

for _, row in sample_files_for_schema.iterrows():
    p = row["file_path"]

    try:
        df_sample = read_cached_chain_file(p, nrows=5)

        schema_rows.append({
            "relative_path": row["relative_path"],
            "suffix": row["suffix"],
            "num_columns": len(df_sample.columns),
            "columns": list(df_sample.columns),
            "read_ok": True,
            "error": "",
        })

    except Exception as e:
        schema_rows.append({
            "relative_path": row["relative_path"],
            "suffix": row["suffix"],
            "num_columns": np.nan,
            "columns": [],
            "read_ok": False,
            "error": str(e),
        })

schema_inspection_df = pd.DataFrame(schema_rows)

display(schema_inspection_df)

print("Read OK counts:")
display(schema_inspection_df["read_ok"].value_counts().rename("count").to_frame())

In [ ]:
# ============================================================
# Identify likely option-chain columns
# ============================================================

all_columns = list(sample_chain_df.columns)
lower_col_map = {str(c).lower(): c for c in all_columns}

def find_first_existing_column(possible_names):
    for name in possible_names:
        if name.lower() in lower_col_map:
            return lower_col_map[name.lower()]
    return None


LIKELY_TRADE_DATE_COL = find_first_existing_column([
    "trade_date",
    "date",
    "quote_date",
    "ms_of_day",
])

LIKELY_EXPIRY_COL = find_first_existing_column([
    "expiration",
    "expiration_date",
    "expiry",
    "expiry_date",
    "exp",
])

LIKELY_STRIKE_COL = find_first_existing_column([
    "strike",
    "strike_price",
])

LIKELY_RIGHT_COL = find_first_existing_column([
    "right",
    "option_type",
    "put_call",
    "cp",
])

LIKELY_BID_COL = find_first_existing_column([
    "bid",
])

LIKELY_ASK_COL = find_first_existing_column([
    "ask",
])

print("Likely trade/date column:", LIKELY_TRADE_DATE_COL)
print("Likely expiry column:", LIKELY_EXPIRY_COL)
print("Likely strike column:", LIKELY_STRIKE_COL)
print("Likely right column:", LIKELY_RIGHT_COL)
print("Likely bid column:", LIKELY_BID_COL)
print("Likely ask column:", LIKELY_ASK_COL)

if LIKELY_EXPIRY_COL is not None:
    print("\nExpiry sample:")
    display(sample_chain_df[LIKELY_EXPIRY_COL].drop_duplicates().head(20))

if LIKELY_STRIKE_COL is not None:
    print("\nStrike sample:")
    display(sample_chain_df[LIKELY_STRIKE_COL].drop_duplicates().sort_values().head(20))

if LIKELY_RIGHT_COL is not None:
    print("\nRight sample:")
    display(sample_chain_df[LIKELY_RIGHT_COL].drop_duplicates().head(20))

In [ ]:
# ============================================================
# Stage 4: Build ThetaData cache index from filenames
# ============================================================

import re

# Recreate chain_files if needed
if "chain_files" not in globals():
    chain_files = sorted([
        p for p in RAW_CHAIN_CACHE_DIR.rglob("*")
        if p.is_file()
    ])

def parse_chain_cache_filename(path):
    """
    Expected pattern:
        SPX_20260615_20260618_160000.pkl

    Returns:
        symbol, trade_date, expiration, quote_time
    """
    path = Path(path)
    name = path.name

    pattern = r"^(?P<symbol>[A-Za-z0-9]+)_(?P<trade_date>\d{8})_(?P<expiration>\d{8})_(?P<quote_time>\d{6})\.pkl$"
    match = re.match(pattern, name)

    if match is None:
        return {
            "parse_ok": False,
            "symbol": None,
            "trade_date": np.nan,
            "expiration": np.nan,
            "quote_time": None,
            "parse_error": "filename_does_not_match_expected_pattern",
        }

    return {
        "parse_ok": True,
        "symbol": match.group("symbol"),
        "trade_date": int(match.group("trade_date")),
        "expiration": int(match.group("expiration")),
        "quote_time": match.group("quote_time"),
        "parse_error": "",
    }


cache_index_rows = []

for p in chain_files:
    parsed = parse_chain_cache_filename(p)
    stat = p.stat()

    cache_index_rows.append({
        "file_path": str(p),
        "file_name": p.name,
        "suffix": p.suffix.lower(),
        "size_mb": stat.st_size / (1024 * 1024),
        **parsed,
    })

cache_index_df = pd.DataFrame(cache_index_rows)

print("Cache files:", len(cache_index_df))
print("Parsed OK:", cache_index_df["parse_ok"].sum())
print("Parse failures:", (~cache_index_df["parse_ok"]).sum())

print("\nSymbol counts:")
display(cache_index_df["symbol"].value_counts(dropna=False).rename("count").to_frame())

print("\nTrade date range:")
print(cache_index_df["trade_date"].min(), "to", cache_index_df["trade_date"].max())

print("\nExpiration range:")
print(cache_index_df["expiration"].min(), "to", cache_index_df["expiration"].max())

print("\nQuote-time counts:")
display(cache_index_df["quote_time"].value_counts(dropna=False).rename("count").to_frame())

display(cache_index_df.head())
display(cache_index_df.tail())

In [ ]:
# ============================================================
# Expiry mapping helpers
# ============================================================

def yyyymmdd_int_to_ts(date_int):
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d")


def calendar_day_diff(later_date_int, earlier_date_int):
    later_ts = yyyymmdd_int_to_ts(later_date_int)
    earlier_ts = yyyymmdd_int_to_ts(earlier_date_int)
    return int((later_ts - earlier_ts).days)


def select_closest_cached_expiry_for_trade_date(
    cache_index,
    trade_date,
    target_expiry_date,
):
    """
    Select listed expiry closest to target_expiry_date for a given trade_date.
    Tie-break: choose later expiry.
    """
    available = cache_index[
        (cache_index["parse_ok"])
        & (cache_index["trade_date"] == int(trade_date))
        & (cache_index["size_mb"] > 0)
    ].copy()

    if available.empty:
        return {
            "selected_expiry_date": np.nan,
            "selected_chain_file": "",
            "selected_symbol": "",
            "selected_quote_time": "",
            "expiry_distance_days": np.nan,
            "num_available_expiries_for_trade_date": 0,
            "expiry_mapping_status": "no_cached_chain_for_trade_date",
        }

    available["expiry_distance_days"] = available["expiration"].apply(
        lambda x: abs(calendar_day_diff(x, target_expiry_date))
    )

    # Tie-break: distance ascending, expiration descending.
    # If duplicate files for same expiry, prefer larger file.
    available = available.sort_values(
        ["expiry_distance_days", "expiration", "size_mb"],
        ascending=[True, False, False],
    )

    selected = available.iloc[0]

    return {
        "selected_expiry_date": int(selected["expiration"]),
        "selected_chain_file": selected["file_path"],
        "selected_symbol": selected["symbol"],
        "selected_quote_time": selected["quote_time"],
        "expiry_distance_days": int(selected["expiry_distance_days"]),
        "num_available_expiries_for_trade_date": available["expiration"].nunique(),
        "expiry_mapping_status": "mapped",
    }

In [ ]:
# ============================================================
# Apply expiry mapping to raw trade candidates
# ============================================================

# Reload raw candidates if needed
if "trade_candidate_raw_df" not in globals():
    trade_candidate_raw_df = pd.read_parquet(TRADE_CANDIDATE_RAW_PARQUET).copy()

expiry_mapping_rows = []

for _, row in trade_candidate_raw_df.iterrows():
    mapping = select_closest_cached_expiry_for_trade_date(
        cache_index=cache_index_df,
        trade_date=row["trade_date"],
        target_expiry_date=row["target_expiry_date"],
    )

    expiry_mapping_rows.append(mapping)

expiry_mapping_df = pd.DataFrame(expiry_mapping_rows)

# Drop placeholder columns before joining real mapped values
placeholder_cols_to_drop = [
    "selected_expiry_date",
    "actual_dte",
    "short_strike_selected",
]

trade_candidate_base_df = trade_candidate_raw_df.drop(
    columns=[c for c in placeholder_cols_to_drop if c in trade_candidate_raw_df.columns]
).copy()

trade_candidate_mapped_df = pd.concat(
    [
        trade_candidate_base_df.reset_index(drop=True),
        expiry_mapping_df.reset_index(drop=True),
    ],
    axis=1,
)

# Calculate actual DTE only for mapped rows
trade_candidate_mapped_df["actual_dte"] = np.nan

mapped_mask = trade_candidate_mapped_df["expiry_mapping_status"] == "mapped"

trade_candidate_mapped_df.loc[mapped_mask, "actual_dte"] = (
    trade_candidate_mapped_df.loc[mapped_mask]
    .apply(
        lambda x: calendar_day_diff(
            x["selected_expiry_date"],
            x["trade_date"],
        ),
        axis=1,
    )
)

print("Rows:", len(trade_candidate_mapped_df))

print("\nExpiry mapping status:")
display(
    trade_candidate_mapped_df["expiry_mapping_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nExpiry distance days:")
display(trade_candidate_mapped_df["expiry_distance_days"].describe())

print("\nActual DTE by side:")
display(
    trade_candidate_mapped_df
    .groupby("trade_side")["actual_dte"]
    .describe()
)

display(
    trade_candidate_mapped_df[
        [
            "trade_date",
            "trade_side",
            "selected_tenor",
            "target_expiry_date",
            "selected_expiry_date",
            "expiry_distance_days",
            "actual_dte",
            "selected_chain_file",
            "spx_close",
            "signal_implied_vol",
            "short_strike_raw",
        ]
    ].tail(40)
)

In [ ]:
# ============================================================
# Strike mapping helpers
# ============================================================

CHAIN_DF_CACHE = {}

def get_cached_chain_df(file_path):
    """
    Read a cached chain file once and reuse it.
    """
    if file_path in CHAIN_DF_CACHE:
        return CHAIN_DF_CACHE[file_path]

    df = read_cached_chain_file(file_path).copy()
    CHAIN_DF_CACHE[file_path] = df

    return df


def raw_2sd_call_strike_actual_dte(spx_close, implied_vol, actual_dte):
    """
    Recalculate call strike using actual listed expiry DTE.
    """
    if pd.isna(spx_close) or pd.isna(implied_vol) or pd.isna(actual_dte):
        return np.nan

    implied_vol_decimal = float(implied_vol) / 100.0
    t = float(actual_dte) / 365.0

    return float(spx_close) * (1.0 + 2.0 * implied_vol_decimal * np.sqrt(t))


def select_listed_short_strike_for_candidate(row):
    """
    For puts:
        selected strike = highest listed put strike <= floor(SPX)

    For calls:
        selected strike = lowest listed call strike >= raw 2SD call strike
        using actual DTE.
    """
    if row["expiry_mapping_status"] != "mapped":
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": "skipped_due_to_unmapped_expiry",
        }

    file_path = row["selected_chain_file"]
    option_right = row["option_right"]

    try:
        chain_df = get_cached_chain_df(file_path)
    except Exception as e:
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": f"chain_read_error: {e}",
        }

    required_cols = ["right", "strike"]

    missing_cols = [c for c in required_cols if c not in chain_df.columns]
    if missing_cols:
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": f"missing_chain_columns: {missing_cols}",
        }

    right_chain_df = chain_df[
        chain_df["right"].astype(str).str.upper() == str(option_right).upper()
    ].copy()

    if right_chain_df.empty:
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": "no_strikes_for_option_right",
        }

    strikes = (
        pd.to_numeric(right_chain_df["strike"], errors="coerce")
        .dropna()
        .drop_duplicates()
        .sort_values()
        .to_numpy()
    )

    if len(strikes) == 0:
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": "no_numeric_strikes_for_option_right",
        }

    if row["trade_side"] == "put":
        raw_strike = raw_atm_put_strike(row["spx_close"])

        eligible = strikes[strikes <= raw_strike]

        if len(eligible) == 0:
            return {
                "short_strike_raw_actual_dte": raw_strike,
                "short_strike_selected": np.nan,
                "strike_mapping_status": "no_put_strike_lte_raw",
            }

        selected_strike = eligible.max()

        return {
            "short_strike_raw_actual_dte": raw_strike,
            "short_strike_selected": selected_strike,
            "strike_mapping_status": "mapped",
        }

    if row["trade_side"] == "call":
        raw_strike = raw_2sd_call_strike_actual_dte(
            spx_close=row["spx_close"],
            implied_vol=row["signal_implied_vol"],
            actual_dte=row["actual_dte"],
        )

        eligible = strikes[strikes >= raw_strike]

        if len(eligible) == 0:
            return {
                "short_strike_raw_actual_dte": raw_strike,
                "short_strike_selected": np.nan,
                "strike_mapping_status": "no_call_strike_gte_raw",
            }

        selected_strike = eligible.min()

        return {
            "short_strike_raw_actual_dte": raw_strike,
            "short_strike_selected": selected_strike,
            "strike_mapping_status": "mapped",
        }

    return {
        "short_strike_raw_actual_dte": np.nan,
        "short_strike_selected": np.nan,
        "strike_mapping_status": "unknown_trade_side",
    }

In [ ]:
# ============================================================
# Apply strike mapping
# ============================================================

strike_mapping_rows = []

for i, row in trade_candidate_mapped_df.iterrows():
    if i % 250 == 0:
        print(f"Processing strike mapping {i}/{len(trade_candidate_mapped_df)}")

    strike_mapping_rows.append(
        select_listed_short_strike_for_candidate(row)
    )

strike_mapping_df = pd.DataFrame(strike_mapping_rows)

trade_candidate_final_df = pd.concat(
    [
        trade_candidate_mapped_df.reset_index(drop=True),
        strike_mapping_df.reset_index(drop=True),
    ],
    axis=1,
)

# For downstream clarity, use actual-DTE raw strike as final raw strike.
trade_candidate_final_df["short_strike_raw_target_tenor"] = (
    trade_candidate_final_df["short_strike_raw"]
)

trade_candidate_final_df["short_strike_raw"] = (
    trade_candidate_final_df["short_strike_raw_actual_dte"]
)

trade_candidate_final_df["mapping_status"] = np.where(
    (trade_candidate_final_df["expiry_mapping_status"] == "mapped")
    & (trade_candidate_final_df["strike_mapping_status"] == "mapped"),
    "mapped",
    "not_mapped",
)

print("Rows:", len(trade_candidate_final_df))

print("\nOverall mapping status:")
display(
    trade_candidate_final_df["mapping_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nExpiry mapping status:")
display(
    trade_candidate_final_df["expiry_mapping_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nStrike mapping status:")
display(
    trade_candidate_final_df["strike_mapping_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nMapped candidates by side:")
display(
    trade_candidate_final_df
    .groupby(["trade_side", "mapping_status"])
    .size()
    .unstack(fill_value=0)
)

display(
    trade_candidate_final_df[
        [
            "trade_date",
            "trade_side",
            "selected_tenor",
            "target_expiry_date",
            "selected_expiry_date",
            "actual_dte",
            "spx_close",
            "signal_implied_vol",
            "short_strike_raw_target_tenor",
            "short_strike_raw",
            "short_strike_selected",
            "mapping_status",
            "expiry_mapping_status",
            "strike_mapping_status",
        ]
    ].tail(50)
)

In [ ]:
# ============================================================
# Save final mapped trade candidate panel
# ============================================================

TRADE_CANDIDATE_PANEL_CSV = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.csv"
TRADE_CANDIDATE_PANEL_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.parquet"

TRADE_CANDIDATE_MAPPING_AUDIT_CSV = AUDIT_DIR / "trade_candidate_mapping_audit_v0_1.csv"

trade_candidate_final_df.to_csv(TRADE_CANDIDATE_PANEL_CSV, index=False)
trade_candidate_final_df.to_parquet(TRADE_CANDIDATE_PANEL_PARQUET, index=False)

mapping_audit_df = (
    trade_candidate_final_df
    .groupby(
        [
            "trade_side",
            "mapping_status",
            "expiry_mapping_status",
            "strike_mapping_status",
        ]
    )
    .size()
    .reset_index(name="count")
)

mapping_audit_df.to_csv(TRADE_CANDIDATE_MAPPING_AUDIT_CSV, index=False)

print("Saved trade candidate CSV:", TRADE_CANDIDATE_PANEL_CSV)
print("Saved trade candidate parquet:", TRADE_CANDIDATE_PANEL_PARQUET)
print("Saved mapping audit:", TRADE_CANDIDATE_MAPPING_AUDIT_CSV)

display(mapping_audit_df)

In [ ]:
# ============================================================
# Final QA: expiry and DTE mapping
# ============================================================

trade_candidate_final_df["dte_minus_selected_tenor"] = (
    trade_candidate_final_df["actual_dte"]
    - trade_candidate_final_df["selected_tenor"]
)

print("Actual DTE minus selected tenor:")
display(trade_candidate_final_df["dte_minus_selected_tenor"].describe())

print("\nExpiry distance days:")
display(trade_candidate_final_df["expiry_distance_days"].describe())

print("\nDTE difference by side:")
display(
    trade_candidate_final_df
    .groupby("trade_side")["dte_minus_selected_tenor"]
    .describe()
)

print("\nLargest absolute DTE differences:")
display(
    trade_candidate_final_df
    .assign(abs_dte_diff=lambda x: x["dte_minus_selected_tenor"].abs())
    .sort_values("abs_dte_diff", ascending=False)
    [
        [
            "trade_date",
            "trade_side",
            "selected_tenor",
            "target_expiry_date",
            "selected_expiry_date",
            "actual_dte",
            "dte_minus_selected_tenor",
            "expiry_distance_days",
        ]
    ]
    .head(30)
)

In [ ]:
# ============================================================
# Final QA: strike moneyness
# ============================================================

trade_candidate_final_df["short_strike_moneyness"] = (
    trade_candidate_final_df["short_strike_selected"]
    / trade_candidate_final_df["spx_close"]
)

trade_candidate_final_df["short_strike_pct_otm"] = np.where(
    trade_candidate_final_df["trade_side"] == "put",
    1.0 - trade_candidate_final_df["short_strike_moneyness"],
    trade_candidate_final_df["short_strike_moneyness"] - 1.0,
)

print("Short strike moneyness by side:")
display(
    trade_candidate_final_df
    .groupby("trade_side")["short_strike_moneyness"]
    .describe()
)

print("\nShort strike pct OTM by side:")
display(
    trade_candidate_final_df
    .groupby("trade_side")["short_strike_pct_otm"]
    .describe()
)

print("\nPut strike examples:")
display(
    trade_candidate_final_df[
        trade_candidate_final_df["trade_side"] == "put"
    ]
    [
        [
            "trade_date",
            "selected_tenor",
            "actual_dte",
            "spx_close",
            "short_strike_raw",
            "short_strike_selected",
            "short_strike_moneyness",
            "short_strike_pct_otm",
        ]
    ]
    .tail(20)
)

print("\nCall strike examples:")
display(
    trade_candidate_final_df[
        trade_candidate_final_df["trade_side"] == "call"
    ]
    [
        [
            "trade_date",
            "selected_tenor",
            "actual_dte",
            "spx_close",
            "signal_implied_vol",
            "short_strike_raw",
            "short_strike_selected",
            "short_strike_moneyness",
            "short_strike_pct_otm",
        ]
    ]
    .tail(20)
)

In [ ]:
# ============================================================
# Final QA: call 2SD strike reasonableness
# ============================================================

call_candidate_df = trade_candidate_final_df[
    trade_candidate_final_df["trade_side"] == "call"
].copy()

call_candidate_df["implied_vol_decimal"] = (
    call_candidate_df["signal_implied_vol"] / 100.0
)

call_candidate_df["expected_2sd_pct_move"] = (
    2.0
    * call_candidate_df["implied_vol_decimal"]
    * np.sqrt(call_candidate_df["actual_dte"] / 365.0)
)

call_candidate_df["selected_call_pct_otm"] = (
    call_candidate_df["short_strike_selected"] / call_candidate_df["spx_close"]
    - 1.0
)

call_candidate_df["selected_minus_raw_strike"] = (
    call_candidate_df["short_strike_selected"]
    - call_candidate_df["short_strike_raw"]
)

print("Expected 2SD pct move:")
display(call_candidate_df["expected_2sd_pct_move"].describe())

print("\nSelected call pct OTM:")
display(call_candidate_df["selected_call_pct_otm"].describe())

print("\nSelected strike minus raw strike:")
display(call_candidate_df["selected_minus_raw_strike"].describe())

print("\nLargest selected strike rounding gaps:")
display(
    call_candidate_df
    .sort_values("selected_minus_raw_strike", ascending=False)
    [
        [
            "trade_date",
            "selected_tenor",
            "actual_dte",
            "spx_close",
            "signal_implied_vol",
            "expected_2sd_pct_move",
            "short_strike_raw",
            "short_strike_selected",
            "selected_minus_raw_strike",
            "selected_call_pct_otm",
        ]
    ]
    .head(30)
)

In [ ]:
# ============================================================
# Final save with QA fields
# ============================================================

TRADE_CANDIDATE_PANEL_CSV = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.csv"
TRADE_CANDIDATE_PANEL_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.parquet"

TRADE_CANDIDATE_QA_CSV = AUDIT_DIR / "trade_candidate_qa_v0_1.csv"

trade_candidate_final_df.to_csv(TRADE_CANDIDATE_PANEL_CSV, index=False)
trade_candidate_final_df.to_parquet(TRADE_CANDIDATE_PANEL_PARQUET, index=False)

trade_candidate_qa_df = pd.DataFrame({
    "metric": [
        "rows",
        "mapped_rows",
        "put_rows",
        "call_rows",
        "min_actual_dte",
        "max_actual_dte",
        "mean_expiry_distance_days",
        "max_expiry_distance_days",
        "mean_put_pct_otm",
        "mean_call_pct_otm",
    ],
    "value": [
        len(trade_candidate_final_df),
        (trade_candidate_final_df["mapping_status"] == "mapped").sum(),
        (trade_candidate_final_df["trade_side"] == "put").sum(),
        (trade_candidate_final_df["trade_side"] == "call").sum(),
        trade_candidate_final_df["actual_dte"].min(),
        trade_candidate_final_df["actual_dte"].max(),
        trade_candidate_final_df["expiry_distance_days"].mean(),
        trade_candidate_final_df["expiry_distance_days"].max(),
        trade_candidate_final_df.loc[
            trade_candidate_final_df["trade_side"] == "put",
            "short_strike_pct_otm",
        ].mean(),
        trade_candidate_final_df.loc[
            trade_candidate_final_df["trade_side"] == "call",
            "short_strike_pct_otm",
        ].mean(),
    ],
})

trade_candidate_qa_df.to_csv(TRADE_CANDIDATE_QA_CSV, index=False)

print("Saved final trade candidate CSV:", TRADE_CANDIDATE_PANEL_CSV)
print("Saved final trade candidate parquet:", TRADE_CANDIDATE_PANEL_PARQUET)
print("Saved candidate QA:", TRADE_CANDIDATE_QA_CSV)

display(trade_candidate_qa_df)